# 2: Chunking Strategies
## RAG-Based Medical QA System — PubMedQA
---
**What this notebook does:**
- Implements Fixed-size chunking
- Implements Recursive chunking (LangChain)
- Compares both strategies with statistics
- Saves chunks for both strategies separately
- Output: `chunks_fixed.json` and `chunks_recursive.json`


## 2.1 Install & Import

In [ ]:
!pip install langchain langchain-text-splitters tiktoken pandas tqdm -q
print('Libraries installed!')

Libraries installed!


In [ ]:
import json
import re
import pandas as pd
from tqdm import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter

print('All imports successful!')

All imports successful!


## 2.2 Upload & Load Your Corpus
Upload `pubmed_corpus.json` from your computer when prompted.

In [ ]:
from google.colab import files
print('Please upload pubmed_corpus.json')
uploaded = files.upload()

Please upload pubmed_corpus.json


Saving pubmed_corpus.json to pubmed_corpus.json


In [ ]:
with open('pubmed_corpus.json', 'r', encoding='utf-8') as f:
    corpus_docs = json.load(f)

print(f'Corpus loaded: {len(corpus_docs)} documents')
print(f'Sample doc keys: {list(corpus_docs[0].keys())}')

Corpus loaded: 5000 documents
Sample doc keys: ['doc_id', 'pubid', 'question', 'abstract', 'long_answer', 'word_count']


## 2.3 Strategy 1: Fixed-Size Chunking

**How it works:**
- Split text every N words (we use 200 words)
- Overlap of 50 words between consecutive chunks
- Overlap ensures context is not lost at chunk boundaries

**Why we use it:** Simple, fast, predictable chunk sizes. Good baseline.

In [ ]:
def fixed_size_chunker(text, chunk_size=200, overlap=50):
    """
    Split text into fixed-size word chunks with overlap.

    Args:
        text: input string
        chunk_size: number of words per chunk
        overlap: number of overlapping words between chunks
    Returns:
        list of chunk strings
    """
    words = text.split()
    chunks = []
    start = 0
    step = chunk_size - overlap  # how far we move forward each time

    while start < len(words):
        end = start + chunk_size
        chunk = ' '.join(words[start:end])
        if len(chunk.split()) >= 20:  # skip very short trailing chunks
            chunks.append(chunk)
        start += step

    return chunks


def build_fixed_chunks(corpus_docs, chunk_size=200, overlap=50):
    """
    Apply fixed chunking to all documents.
    Returns list of chunk dicts with metadata.
    """
    all_chunks = []
    chunk_index = 0

    for doc in tqdm(corpus_docs, desc='Fixed chunking'):
        text = doc['abstract']
        chunks = fixed_size_chunker(text, chunk_size=chunk_size, overlap=overlap)

        for i, chunk_text in enumerate(chunks):
            all_chunks.append({
                'chunk_id'    : f'fixed_{chunk_index:06d}',
                'doc_id'      : doc['doc_id'],
                'pubid'       : doc['pubid'],
                'chunk_index' : i,
                'text'        : chunk_text,
                'word_count'  : len(chunk_text.split()),
                'strategy'    : 'fixed',
                'source_question': doc['question']  # store original question for reference
            })
            chunk_index += 1

    return all_chunks


# Run fixed chunking
fixed_chunks = build_fixed_chunks(corpus_docs, chunk_size=200, overlap=50)

print(f'\nFixed Chunking Results:')
print(f'  Total chunks : {len(fixed_chunks)}')
print(f'  Avg words/chunk: {sum(c["word_count"] for c in fixed_chunks)/len(fixed_chunks):.1f}')

Fixed chunking: 100%|██████████| 5000/5000 [00:00<00:00, 17618.30it/s]


Fixed Chunking Results:
  Total chunks : 8771
  Avg words/chunk: 133.8


## 2.4 Strategy 2: Recursive Character Chunking

**How it works:**
- Uses LangChain's `RecursiveCharacterTextSplitter`
- Tries to split by: paragraph → sentence → word → character (in order)
- Respects sentence boundaries much better than fixed chunking
- Uses token count (not word count) for sizing

**Why we use it:** More semantically coherent chunks. Better for medical text.

In [ ]:
# Initialize the recursive splitter
# chunk_size=800 characters (~150-200 words for medical text)
# chunk_overlap=150 characters

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=['\n\n', '\n', '. ', '? ', '! ', ' ', ''],
    length_function=len,
)


def build_recursive_chunks(corpus_docs, splitter):
    """
    Apply recursive chunking to all documents.
    Returns list of chunk dicts with metadata.
    """
    all_chunks = []
    chunk_index = 0

    for doc in tqdm(corpus_docs, desc='Recursive chunking'):
        text = doc['abstract']
        chunks = splitter.split_text(text)

        for i, chunk_text in enumerate(chunks):
            chunk_text = chunk_text.strip()
            if len(chunk_text.split()) < 20:  # skip very short chunks
                continue

            all_chunks.append({
                'chunk_id'    : f'recursive_{chunk_index:06d}',
                'doc_id'      : doc['doc_id'],
                'pubid'       : doc['pubid'],
                'chunk_index' : i,
                'text'        : chunk_text,
                'word_count'  : len(chunk_text.split()),
                'char_count'  : len(chunk_text),
                'strategy'    : 'recursive',
                'source_question': doc['question']
            })
            chunk_index += 1

    return all_chunks


# Run recursive chunking
recursive_chunks = build_recursive_chunks(corpus_docs, recursive_splitter)

print(f'\nRecursive Chunking Results:')
print(f'  Total chunks : {len(recursive_chunks)}')
print(f'  Avg words/chunk: {sum(c["word_count"] for c in recursive_chunks)/len(recursive_chunks):.1f}')

Recursive chunking: 100%|██████████| 5000/5000 [00:00<00:00, 17347.10it/s]


Recursive Chunking Results:
  Total chunks : 11605
  Avg words/chunk: 92.6


## 2.5 Compare Both Strategies


In [ ]:
import numpy as np

def chunk_stats(chunks, name):
    wc = [c['word_count'] for c in chunks]
    return {
        'Strategy'        : name,
        'Total Chunks'    : len(chunks),
        'Min Words'       : min(wc),
        'Max Words'       : max(wc),
        'Mean Words'      : round(np.mean(wc), 1),
        'Median Words'    : round(np.median(wc), 1),
        'Std Dev'         : round(np.std(wc), 1),
    }

stats_fixed     = chunk_stats(fixed_chunks, 'Fixed (200w, 50 overlap)')
stats_recursive = chunk_stats(recursive_chunks, 'Recursive (800c, 150 overlap)')

comparison_df = pd.DataFrame([stats_fixed, stats_recursive])
comparison_df = comparison_df.set_index('Strategy')

print('=== CHUNKING STRATEGY COMPARISON ===')
print(comparison_df.to_string())
print('\n(Save this table for your Ablation Study in the report!)')

=== CHUNKING STRATEGY COMPARISON ===
                               Total Chunks  Min Words  Max Words  Mean Words  Median Words  Std Dev
Strategy                                                                                            
Fixed (200w, 50 overlap)               8771         20        200       133.8         150.0     63.2
Recursive (800c, 150 overlap)         11605         20        172        92.6         100.0     25.9

(Save this table for your Ablation Study in the report!)


## 2.6 Visual Inspection — Sample Chunks

In [ ]:
# Compare how the same document is chunked by each strategy
sample_doc_id = corpus_docs[0]['doc_id']

fixed_sample = [c for c in fixed_chunks if c['doc_id'] == sample_doc_id]
recursive_sample = [c for c in recursive_chunks if c['doc_id'] == sample_doc_id]

print(f'=== SAME DOCUMENT: {sample_doc_id} ===')
print(f'Original abstract ({corpus_docs[0]["word_count"]} words):')
print(f'{corpus_docs[0]["abstract"][:200]}...\n')

print(f'--- FIXED: produced {len(fixed_sample)} chunks ---')
for i, c in enumerate(fixed_sample):
    print(f'  Chunk {i+1} ({c["word_count"]} words): {c["text"][:120]}...')

print(f'\n--- RECURSIVE: produced {len(recursive_sample)} chunks ---')
for i, c in enumerate(recursive_sample):
    print(f'  Chunk {i+1} ({c["word_count"]} words): {c["text"][:120]}...')

=== SAME DOCUMENT: pubmed_00000 ===
Original abstract (254 words):
Although the use of alternative medicine in the United States is increasing, no published studies have documented the effectiveness of naturopathy for treatment of menopausal symptoms compared to wome...

--- FIXED: produced 2 chunks ---
  Chunk 1 (200 words): Although the use of alternative medicine in the United States is increasing, no published studies have documented the ef...
  Chunk 2 (104 words): (57.0% versus 33.1%), and hot flashes (69.6% versus 55.6%) at baseline than those who received conventional treatment. I...

--- RECURSIVE: produced 3 chunks ---
  Chunk 1 (110 words): Although the use of alternative medicine in the United States is increasing, no published studies have documented the ef...
  Chunk 2 (104 words): . Improvement in selected menopausal symptoms. In univariate analyses, patients treated with naturopathy for menopausal ...
  Chunk 3 (47 words): . Naturopathy patients reported improvement for

## 2.7 Save Chunks to JSON

In [ ]:
# Save fixed chunks
with open('chunks_fixed.json', 'w', encoding='utf-8') as f:
    json.dump(fixed_chunks, f, indent=2, ensure_ascii=False)
print(f'Fixed chunks saved   → chunks_fixed.json ({len(fixed_chunks)} chunks)')

# Save recursive chunks
with open('chunks_recursive.json', 'w', encoding='utf-8') as f:
    json.dump(recursive_chunks, f, indent=2, ensure_ascii=False)
print(f'Recursive chunks saved → chunks_recursive.json ({len(recursive_chunks)} chunks)')

print('\nStep 2 Complete!')
print('Next: Step 3 — Generate Embeddings & Upload to Pinecone')

Fixed chunks saved   → chunks_fixed.json (8771 chunks)
Recursive chunks saved → chunks_recursive.json (11605 chunks)

Step 2 Complete!
Next: Step 3 — Generate Embeddings & Upload to Pinecone


## 2.8 Download Files

In [ ]:
from google.colab import files
files.download('chunks_fixed.json')
files.download('chunks_recursive.json')
print('Downloaded!')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded!
